# Length Experiment: How does candidate quality vary with protein length under a fixed RSO budget?

Iterates lengths [100, 150, 200] (quick profile) or [100,150,200,300] (full) × seeds. **Supports resumption:** if a run directory already has status.status == 'completed', it is SKIPPED entirely; if 'failed' or partially written, only missing stages are re-run.

**GPU warning:** 300-aa with AF2 validation may require >24 GB GPU RAM. If OOM occurs, drop length=300 from configs/length_experiment_full.yaml and resume.


In [ ]:
# --- setup: check GPU, install deps, pick PROFILE (run once per session) ---
import os, shutil, subprocess, sys, json, time, pathlib, datetime
from pathlib import Path

#@markdown Choose experiment profile — quick = 3 lengths × fewer seeds, full = 4 lengths × more seeds
PROFILE = "quick"  #@param ["quick", "full"] {type:"string"}

GPU_AVAILABLE = False
try:
    import subprocess as sp
    r = sp.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
               capture_output=True, text=True, timeout=20)
    if r.returncode == 0 and r.stdout.strip():
        GPU_AVAILABLE = True
        print("GPU DETECTED:", r.stdout.strip())
except Exception as e:
    print("No nvidia-smi:", e)

if not GPU_AVAILABLE:
    print("FATAL: No NVIDIA GPU. Runtime -> Change runtime type -> Hardware accelerator -> GPU.")
    print("Refusing to run computationally expensive steps on CPU.")
    # raise SystemExit(1)

PROJECT = "/content/rso-protein-design-exploration"
if not Path(PROJECT).exists():
    get_ipython().system('git clone --depth 1 https://github.com/sokrypton/ColabDesign.git 2>&1 | tail -5')
    print("Cloning your project repo (adjust URL if needed):")
    # get_ipython().system('git clone https://github.com/YOUR_USER/rso-protein-design-exploration.git {PROJECT} 2>&1 | tail -3')

sys.path.insert(0, "/content/ColabDesign")
get_ipython().run_line_magic('pip', 'install -q \
  git+https://github.com/sokrypton/ColabDesign.git@main biopython tqdm pyyaml pandas matplotlib seaborn 2>&1 | tail -5')

import numpy as np, pandas as pd, random
import jax, jax.numpy as jnp
print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())
assert any("gpu" in str(d).lower() or "cuda" in str(d).lower() for d in jax.devices()), "JAX sees no GPU."

# JAX >=0.11 compatibility shim for ColabDesign main (as of 2026-09,
# ColabDesign calls jax.lib.xla_bridge.get_backend() in clear_mem(), but
# jax.lib.xla_bridge was removed in JAX 0.10+/moved to jax._src.xla_bridge).
if not hasattr(jax.lib, "xla_bridge"):
    from jax._src import xla_bridge as _xla_bridge
    jax.lib.xla_bridge = _xla_bridge
print("xla_bridge backend:", jax.lib.xla_bridge.get_backend().platform)

sys.path.insert(0, str(Path(PROJECT) / "src"))
try:
    from rso_exploration.config import load_config, validate_config
    from rso_exploration.paths import ensure_run_dir, backbone_pdb_path, candidate_fasta_path, candidate_csv_path, predicted_pdb_path, metadata_path, status_path, loss_history_path
    from rso_exploration.provenance import collect_provenance, save_provenance
    print("Local rso_exploration package loaded.")
except Exception as e:
    print("Local package not available, using inline helpers (" + str(e) + ")")


## Experiment plan & resumption logic


In [ ]:
# Load quick / full config, enumerate tasks, and decide resume/skip per run dir

CFG_NAME = f"length_experiment_{PROFILE}.yaml"
cfg_path = Path(PROJECT) / "configs" / CFG_NAME
if cfg_path.exists():
    cfg = load_config(cfg_path)
    errs = validate_config(cfg)
    print(f"Loaded {CFG_NAME}: experiment={cfg.experiment_name}, errors={errs}")
    LENGTHS = list(cfg.experiment.lengths)
    SEEDS   = list(cfg.experiment.seeds)
    RSO_CFG = cfg.rso
    MPNN_CFG = cfg.mpnn
    VAL_CFG  = cfg.validation
else:
    print(f"Config {cfg_path} not found — using hard-coded defaults for profile={PROFILE}.")
    if PROFILE == "quick":
        LENGTHS = [100, 150, 200]
        SEEDS   = [42]
    else:
        LENGTHS = [100, 150, 200, 300]
        SEEDS   = [42, 123, 2024]
    RSO_CFG = MPNN_CFG = VAL_CFG = None

# Build task list: (length, seed, backbone_id, run_id)
RESULTS_ROOT = Path("/content/results/runs")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

tasks = []
for length in LENGTHS:
    for seed in SEEDS:
        run_id = f"len{length}_s{seed}_" + datetime.datetime.utcnow().strftime("%Y%m%d")
        run_dir = RESULTS_ROOT / run_id
        status_file = run_dir / "status.json"
        decision = "TODO"
        stage_done = {"stage1": False, "stage2": False, "stage3": False}
        if status_file.exists():
            try:
                s = json.loads(status_file.read_text())
                cur = s.get("status", "")
                if cur == "completed":
                    decision = "SKIP (completed)"
                    stage_done = {"stage1": True, "stage2": True, "stage3": True}
                else:
                    decision = "RESUME"
                    bb = (s.get("backbones") or {}).get("bb0", {})
                    if bb.get("status") == "rso_done": stage_done["stage1"] = True
                    if "stage2_mpnn" in s: stage_done["stage2"] = True
            except Exception as e:
                decision = "RERUN (bad status.json: " + str(e)[:80] + ")"
        tasks.append({
            "length": length, "seed": seed, "run_id": run_id,
            "run_dir": str(run_dir), "decision": decision, **stage_done
        })

tododf = pd.DataFrame(tasks)
tododf.index.name = "task"
print(f"\nTotal tasks: {len(tasks)}")
print(f"  TODO   : {(tododf.decision == 'TODO').sum()}")
print(f"  RESUME : {(tododf.decision == 'RESUME').sum()}")
print(f"  SKIP   : {(tododf.decision.str.startswith('SKIP')).sum()}")
print(f"  RERUN  : {(tododf.decision.str.startswith('RERUN')).sum()}")
display(tododf)


## Run loop (RSO → MPNN → Validation) for each task


In [ ]:
# Shared helpers (same as notebook 01, inlined so the experiment is self-contained)
from colabdesign import mk_afdesign_model, clear_mem
from colabdesign.mpnn import mk_mpnn_model
from colabdesign.af.alphafold.common import residue_constants

def add_rg_loss(self, weight=0.1):
    def loss_fn(inputs, outputs):
        positions = outputs["structure_module"]["final_atom_positions"]
        ca = positions[:, residue_constants.atom_order["CA"]]
        center = ca.mean(0)
        rg = jnp.sqrt(jnp.square(ca - center).sum(-1).mean() + 1e-8)
        rg_th = 2.38 * ca.shape[0] ** 0.365
        return {"rg": jax.nn.elu(rg - rg_th)}
    self._callbacks["model"]["loss"].append(loss_fn)
    self.opt["weights"]["rg"] = weight

def ca_rmsd(xyz1, xyz2):
    d = xyz1 - xyz2
    return float(np.sqrt(np.mean(np.sum(d*d, axis=-1))))

def tm_score_approx(ref_ca, mod_ca, L=None):
    L = L or len(ref_ca)
    d0 = 1.24 * max(L - 15, 1) ** (1/3) - 1.8
    d = np.sqrt(np.sum((ref_ca - mod_ca) ** 2, axis=-1))
    return float(np.mean(1.0 / (1.0 + (d / d0) ** 2)))

def _paths(rd, bb_id):
    return {
        "pdb":  rd/"stage1_rso"/f"{bb_id}.pdb",
        "loss": rd/"stage1_rso"/f"{bb_id}_loss_history.csv",
        "fasta":rd/"stage2_mpnn"/f"{bb_id}_candidates.fasta",
        "csv":  rd/"stage2_mpnn"/f"{bb_id}_candidates.csv",
        "s3":   rd/"stage3_validation",
        "meta": rd/"metadata.json",
        "stat": rd/"status.json",
    }

overall_start = time.time()
task_results = []

for ti, task in enumerate(tasks):
    LENGTH = task["length"]
    SEED   = task["seed"]
    rd     = Path(task["run_dir"])
    rd.mkdir(parents=True, exist_ok=True)
    (rd/"stage1_rso").mkdir(exist_ok=True)
    (rd/"stage2_mpnn").mkdir(exist_ok=True)
    (rd/"stage3_validation").mkdir(exist_ok=True)
    bb_id = "bb0"
    P = _paths(rd, bb_id)

    print(f"\n========== TASK {ti+1}/{len(tasks)} | L={LENGTH} seed={SEED} | {task['decision']} ==========")
    random.seed(SEED); np.random.seed(SEED)

    status = {}
    if P["stat"].exists():
        try: status = json.loads(P["stat"].read_text())
        except: status = {}
    status.setdefault("backbones", {})
    status["backbones"].setdefault(bb_id, {})

    # ---------------- STAGE 1: RSO backbone ----------------
    if not task["stage1"]:
        print("  [STAGE1] RSO hallucination ...")
        try:
            clear_mem()
            af_model = mk_afdesign_model(protocol="hallucination", loss_callback=None)
            af_model.prep_inputs(length=LENGTH)

            # Per-official-notebook: drop rg weight for >600 aa
            if RSO_CFG is not None:
                w = RSO_CFG.loss
                rg_w = w.rg_weight
                hw, cw, pw, pe = w.helix_weight, w.con_weight, w.plddt_weight, w.pae_weight
                s1, s2 = RSO_CFG.stage1_iterations, RSO_CFG.stage2_iterations
            else:
                rg_w = 0.01 if LENGTH > 600 else (0.001 if LENGTH >= 1000 else 0.1)
                hw, cw, pw, pe = -0.2, 1.0, 0.5, 0.5
                s1, s2 = 90, 10

            add_rg_loss(af_model, rg_w)
            af_model.opt["weights"]["helix"] = hw
            af_model.opt["weights"]["con"]   = cw
            af_model.opt["weights"]["plddt"] = pw
            af_model.opt["weights"]["pae"]   = pe
            af_model.restart(mode=["gumbel","soft"], rm_aa="C")

            t0 = time.time()
            af_model.design_logits(s1)
            af_model.design_logits(s2, save_best=True)
            rso_time = time.time() - t0
            print(f"  [STAGE1] DONE in {rso_time:.1f}s")

            af_model.save_pdb(str(P["pdb"]))
            if hasattr(af_model, "log"):
                ldf = pd.DataFrame(af_model.log)
                ldf.index.name = "step"
                ldf.to_csv(P["loss"])
            best_seq = ""
            try:
                seqs = af_model.get_seqs()
                best_seq = seqs[0] if isinstance(seqs, (list, tuple)) else str(seqs)
            except: pass
            final_loss = float(np.mean(af_model.aux["losses"]["total"])) if 'total' in (af_model.aux.get("losses") or {}) else None

            status["rso_runtime_seconds"] = rso_time
            status["status"] = "stage2_pending"
            status["backbones"][bb_id] = {"status": "rso_done", "final_loss": final_loss,
                                          "length": LENGTH, "seed": SEED,
                                          "best_rso_seq": best_seq, "rg_weight": rg_w}
            P["stat"].write_text(json.dumps(status, indent=2))
        except Exception as e:
            print(f"  [STAGE1] FAILED: {type(e).__name__}: {str(e)[:300]}")
            status["status"] = "failed"
            status["stage1_error"] = f"{type(e).__name__}: {str(e)[:500]}"
            P["stat"].write_text(json.dumps(status, indent=2))
            task_results.append({**task, "outcome": "stage1_failed", "error": str(e)[:200]})
            continue
    else:
        print("  [STAGE1] SKIP — backbone already done.")

    # ---------------- STAGE 2: MPNN ----------------
    pdb_out = P["pdb"]
    if not task["stage2"]:
        print("  [STAGE2] ProteinMPNN design ...")
        try:
            clear_mem()
            NUM_SEQS = MPNN_CFG.num_seqs if MPNN_CFG else 8
            TEMP     = MPNN_CFG.temperature if MPNN_CFG else 0.1
            WEIGHTS  = MPNN_CFG.weights if MPNN_CFG else "soluble"
            RM_AA    = MPNN_CFG.rm_aa if MPNN_CFG else "C"
            mpnn_model = mk_mpnn_model(weights=WEIGHTS)
            mpnn_model.prep_inputs(pdb_filename=str(pdb_out), chain="A", rm_aa=RM_AA)
            t0 = time.time()
            out = mpnn_model.sample(num=NUM_SEQS//8, batch=8, temperature=TEMP)
            mpnn_time = time.time() - t0
            print(f"  [STAGE2] DONE in {mpnn_time:.1f}s — {len(out['seq'])} seq(s)")

            with open(P["fasta"], "w") as f:
                for i, (seq, score) in enumerate(zip(out["seq"], out["score"])):
                    f.write(f">c{i} mpnn_score={float(score):.4f}\n{seq}\n")
            pd.DataFrame({
                "candidate_id": [f"c{i}" for i in range(len(out["seq"]))],
                "seq": out["seq"],
                "score": [float(s) for s in out["score"]],
                "temperature": [TEMP]*len(out["seq"]),
                "weights": [WEIGHTS]*len(out["seq"]),
            }).to_csv(P["csv"], index=False)

            status["stage2_mpnn"] = {"num_seqs": NUM_SEQS, "temperature": TEMP,
                                     "weights": WEIGHTS, "rm_aa": RM_AA,
                                     "mpnn_runtime_seconds": mpnn_time}
            status["status"] = "stage3_pending"
            P["stat"].write_text(json.dumps(status, indent=2))
        except Exception as e:
            print(f"  [STAGE2] FAILED: {type(e).__name__}: {str(e)[:300]}")
            status["status"] = "failed"
            status["stage2_error"] = f"{type(e).__name__}: {str(e)[:500]}"
            P["stat"].write_text(json.dumps(status, indent=2))
            task_results.append({**task, "outcome": "stage2_failed", "error": str(e)[:200]})
            continue
    else:
        print("  [STAGE2] SKIP — MPNN already done.")
        # reload sequences from CSV
        if P["csv"].exists():
            odf = pd.read_csv(P["csv"])
            out = {"seq": list(odf["seq"].values), "score": list(odf["score"].values)}
        else:
            print("  [STAGE2] CSV missing — will try to re-run MPNN ...")
            # (caller can set stage2=False in task dict and re-run if needed)

    # ---------------- STAGE 3: AF2 validation ----------------
    if not task["stage3"]:
        print("  [STAGE3] AF2 single-seq validation ...")
        try:
            clear_mem()
            AF_MODEL_NAME = VAL_CFG.alphafold_model if VAL_CFG else "model_4_ptm"
            NUM_RECYCLES  = VAL_CFG.num_recycles if VAL_CFG else 3
            af_val = mk_afdesign_model(protocol="fixbb", use_templates=False)
            af_val.prep_inputs(pdb_filename=str(pdb_out), chain="A")
            af_val.restart(rm_aa="C")

            rows = []; val_times = []
            for i in range(len(out["seq"])):
                cid = f"c{i}"
                seq = out["seq"][i]
                t0 = time.time()
                af_val.predict(seq=seq, num_recycles=NUM_RECYCLES, num_models=1,
                               models=AF_MODEL_NAME, verbose=False)
                aux = af_val.aux
                log = aux.get("log", {})
                plddt_arr = np.array(aux.get("plddt", log.get("plddt"))).astype(float)
                mean_plddt = float(np.mean(plddt_arr)) if plddt_arr.size else None
                ptm   = float(log["ptm"])  if "ptm"  in log else None
                rmsd  = float(log["rmsd"]) if "rmsd" in log else None
                tm = None
                try:
                    positions = aux["structure_module"]["final_atom_positions"]
                    pred_ca = np.array(positions[:, residue_constants.atom_order["CA"]], dtype=float)
                    ref_ca = []
                    for line in open(pdb_out):
                        if line.startswith("ATOM  ") and line[12:16].strip() == "CA":
                            ref_ca.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
                    ref_ca = np.array(ref_ca, dtype=float)
                    n = min(len(ref_ca), len(pred_ca))
                    if n > 15:
                        tm = tm_score_approx(ref_ca[:n], pred_ca[:n], L=n)
                        if rmsd is None:
                            rmsd = ca_rmsd(ref_ca[:n], pred_ca[:n])
                except: pass
                vt = time.time() - t0; val_times.append(vt)
                (P["s3"] / f"{bb_id}_{cid}_metrics.json").write_text(json.dumps({
                    "rmsd_angstrom": rmsd, "tm_score": tm, "mean_plddt": mean_plddt,
                    "ptm": ptm, "validation_runtime_seconds": vt,
                    "validation_model": AF_MODEL_NAME
                }, indent=2))
                try: af_val.save_pdb(str(P["s3"] / f"{bb_id}_{cid}_predicted.pdb"))
                except: pass
                rows.append({"c": cid, "rmsd": rmsd, "tm": tm, "plddt": mean_plddt, "ptm": ptm, "time_s": vt})
                print(f"    {cid}: RMSD={rmsd}  TM={tm}  pLDDT={mean_plddt}  ({vt:.1f}s)")

            status["status"] = "completed"
            status["validation"] = {"model": AF_MODEL_NAME, "num_recycles": NUM_RECYCLES,
                                     "total_validation_seconds": sum(val_times)}
            status["backbones"][bb_id]["candidates"] = rows
            P["stat"].write_text(json.dumps(status, indent=2, default=str))
            best_rmsd = min((r["rmsd"] for r in rows if r.get("rmsd") is not None), default=None)
            task_results.append({**task, "outcome": "ok", "best_rmsd": best_rmsd,
                                 "num_candidates": len(rows)})
        except Exception as e:
            print(f"  [STAGE3] FAILED: {type(e).__name__}: {str(e)[:300]}")
            status["status"] = "failed"
            status["stage3_error"] = f"{type(e).__name__}: {str(e)[:500]}"
            P["stat"].write_text(json.dumps(status, indent=2, default=str))
            task_results.append({**task, "outcome": "stage3_failed", "error": str(e)[:200]})
            continue
    else:
        print("  [STAGE3] SKIP — validation already done.")
        task_results.append({**task, "outcome": "previously_completed"})

print(f"\n========== ALL TASKS DONE in {time.time()-overall_start:.1f}s ==========")
display(pd.DataFrame(task_results))


## Summary


In [ ]:
# Collect metrics via project script, print summary, zip results
collect_script = Path(PROJECT)/"scripts"/"collect_metrics.py"
if collect_script.exists():
    print(f"Running {collect_script} ...")
    r = subprocess.run([sys.executable, str(collect_script)], capture_output=True, text=True)
    print(r.stdout)
    if r.stderr.strip(): print("STDERR:", r.stderr)
else:
    print("collect_metrics.py not found; no automatic summary.")

metrics_csv = Path(PROJECT)/"results"/"metrics"/"all_candidates.csv"
if metrics_csv.exists():
    mdf = pd.read_csv(metrics_csv)
    print(f"\nAll candidates ({len(mdf)} rows):")
    if "length" in mdf.columns:
        summary = mdf.groupby("length").agg(
            n=("length","count"),
            rmsd_mean=("rmsd_angstrom","mean"),
            rmsd_min=("rmsd_angstrom","min"),
            tm_mean=("tm_score","mean"),
            tm_max=("tm_score","max"),
            plddt_mean=("mean_plddt","mean"),
        ).round(3)
        display(summary)
    else:
        display(mdf.head(20))

    import shutil, datetime
    archive = "/content/length_exp_results_" + datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ") + ".zip"
    base = str(Path(PROJECT)/"results") if Path(PROJECT).exists() else str(RESULTS_ROOT.parent)
    print("\nArchiving", base, "->", archive)
    shutil.make_archive(archive.replace(".zip",""), 'zip', base)
    print(f"Wrote {archive} ({Path(archive).stat().st_size/1e6:.1f} MB)")
    try:
        from google.colab import files
        files.download(archive)
    except Exception:
        print("Not in Colab UI; download archive via file browser.")
